In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None)
# display(df)

In [ ]:
"""
crossfill_opportunity.py
Before building any coalesce logic, measure the RECOVERY OPPORTUNITY:
for each core column, how many currently-missing values could be filled
from sibling / source-specific / enrichment columns?

Built for the 7-source tick corpus (OpenAlex, Lens, EuropePMC, Scopus,
Web of Science, PubMed, Global Index Medicus). Colab Pro.
"""

import pandas as pd
import numpy as np

from google.colab import drive; drive.mount('/content/drive')
PATH = '/content/drive/MyDrive/COGNOSX/TICKs_and_climate_change/preprocessing_data_v2/master_enriched.parquet'
df = pd.read_parquet(PATH)
N = len(df)
cols = list(df.columns)


# ----------------------------------------------------------------------
# Empty-aware missingness (empty list/dict/str count as missing, not just NaN)
# ----------------------------------------------------------------------
def cell_is_empty(x):
    if x is None:
        return True
    if isinstance(x, float) and np.isnan(x):
        return True
    if isinstance(x, (list, np.ndarray, dict, str)):
        return len(x) == 0
    return False

def empty_mask(series):
    return series.map(cell_is_empty)

# cache masks (avoids recomputing per concept)
_mask_cache = {c: empty_mask(df[c]) for c in cols}


# ----------------------------------------------------------------------
# concept -> substrings that identify candidate columns for that concept.
# Tune these once you see your real column names (paste them and I'll adjust).
# ----------------------------------------------------------------------
CORE_CONCEPTS = {
    "doi":         ["doi"],
    "pmid":        ["pmid", "pubmed_id", "pubmedid"],
    "pmcid":       ["pmcid"],
    "title":       ["title"],
    "abstract":    ["abstract"],
    "year":        ["year", "pub_year", "publication_year"],
    "date":        ["date"],
    "authors":     ["author"],
    "journal":     ["journal", "venue", "source_title", "container", "host_venue"],
    "language":    ["language", "lang"],
    "doc_type":    ["document_type", "doctype", "publication_type", "type"],
    "keywords":    ["keyword", "mesh", "concept"],
    "affiliation": ["affil", "institution"],
    "country":     ["country"],
}

# Source tokens used to tell a canonical column from a source-specific donor
SOURCE_TOKENS = [
    "openalex", "oa_", "oa__", "lens", "europepmc", "epmc",
    "scopus", "webofscience", "wos", "pubmed", "pmc",
    "globalindexmedicus", "gim", "medicus",
]

def find_cols(subs):
    return [c for c in cols if any(s in c.lower() for s in subs)]

def is_source_specific(colname):
    return any(tok in colname.lower() for tok in SOURCE_TOKENS)

def pick_target(concept, candidates):
    """Choose the canonical column to FILL; the rest are donors."""
    # 1. exact concept name
    for c in candidates:
        if c.lower() == concept:
            return c
    # 2. a non-source-prefixed column (the merged canonical field)
    non_source = [c for c in candidates if not is_source_specific(c)]
    if non_source:
        # prefer the highest-coverage canonical
        return min(non_source, key=lambda c: _mask_cache[c].sum())
    # 3. fallback: highest-coverage overall
    return min(candidates, key=lambda c: _mask_cache[c].sum())


# ----------------------------------------------------------------------
# Build the opportunity report
# ----------------------------------------------------------------------
rows = []
detail = {}

for concept, subs in CORE_CONCEPTS.items():
    candidates = find_cols(subs)
    if not candidates:
        continue

    target = pick_target(concept, candidates)
    donors = [c for c in candidates if c != target]

    t_missing = _mask_cache[target]          # True where target is empty
    n_missing = int(t_missing.sum())
    cov_now = 100 * (N - n_missing) / N

    # union of rows a donor could fill (target empty AND donor filled)
    recoverable_union = pd.Series(False, index=df.index)
    per_donor = []
    for d in donors:
        d_filled = ~_mask_cache[d]
        fillable = t_missing & d_filled
        k = int(fillable.sum())
        if k > 0:
            per_donor.append((d, k, round(100 * k / max(n_missing, 1), 1)))
            recoverable_union |= fillable

    n_recover = int(recoverable_union.sum())
    cov_after = 100 * (N - n_missing + n_recover) / N

    rows.append({
        "concept": concept,
        "target_col": target,
        "coverage_now_%": round(cov_now, 1),
        "missing": n_missing,
        "recoverable": n_recover,
        "recover_%_of_gap": round(100 * n_recover / max(n_missing, 1), 1),
        "coverage_after_%": round(cov_after, 1),
        "n_donors": len(donors),
    })
    detail[concept] = sorted(per_donor, key=lambda x: -x[1])

report = (pd.DataFrame(rows)
          .sort_values("recoverable", ascending=False)
          .reset_index(drop=True))

pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", 40)
print(f"Records: {N:,}\n")
print("================ CROSS-FILL OPPORTUNITY ================\n")
print(report.to_string(index=False))


# ----------------------------------------------------------------------
# Per-concept donor breakdown (which source fills the gap, and how much)
# ----------------------------------------------------------------------
print("\n================ DONOR BREAKDOWN (top gaps) ================")
for concept in report.query("recoverable > 0")["concept"]:
    print(f"\n[{concept}]  target = {pick_target(concept, find_cols(CORE_CONCEPTS[concept]))}")
    for donor, k, pct in detail[concept]:
        print(f"    {donor:<45} fills {k:>7,}  ({pct}% of gap)")


# ----------------------------------------------------------------------
# DERIVATION probe: year recoverable from any date-like column
# (copy != compute — this is the compute case)
# ----------------------------------------------------------------------
year_targets = find_cols(["year", "publication_year", "pub_year"])
date_donors = find_cols(["date"])
if year_targets and date_donors:
    yt = pick_target("year", year_targets)
    y_missing = _mask_cache[yt]
    derivable = pd.Series(False, index=df.index)
    for dc in date_donors:
        extracted = df[dc].astype(str).str.extract(r"(\b(?:19|20)\d{2}\b)")[0]
        derivable |= (y_missing & extracted.notna())
    print(f"\n[DERIVE] year missing in '{yt}': {int(y_missing.sum()):,} | "
          f"recoverable from date columns: {int(derivable.sum()):,}")


report.to_csv("crossfill_opportunity.csv", index=False)
print("\nSaved -> crossfill_opportunity.csv")

In [ ]:
import pandas as pd

PATH_IN  = '/content/drive/MyDrive/COGNOSX/TICKs_and_climate_change/preprocessing_data_v2/master_enriched.parquet'
PATH_OUT = '/content/drive/MyDrive/COGNOSX/TICKs_and_climate_change/preprocessing_data_v2/master_enriched.csv'

df = pd.read_parquet(PATH_IN)
print(f"Loaded: {len(df):,} rows × {len(df.columns)} columns")

df.to_csv(PATH_OUT, index=False, encoding='utf-8-sig')
print(f"Saved  → {PATH_OUT}")